In [ ]:
from llama_index.core.node_parser import SemanticSplitterNodeParser, SentenceSplitter
import pickle
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import nest_asyncio
nest_asyncio.apply()
from dotenv import load_dotenv
import os
load_dotenv()

embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

with open("/Users/rifah/Desktop/p/species/mds.pkl", "rb") as f:
    docs = pickle.load(f)


splitter = SemanticSplitterNodeParser(buffer_size=1, embed_model=embed_model,include_metadata=True)
tags = ['paper_body', 'bibliography'];
nodes = []

for doc in docs:
    try: 
        chunk = splitter.get_nodes_from_documents([doc])
        nodes.extend(chunk)
    except Exception as e:
        print(e, "Unable to chunk")
    
     

Embedding attempt failed: TextEncodeInput must be Union[TextInputSequence, Tuple[InputSequence, InputSequence]]
Embedding attempt failed: TextEncodeInput must be Union[TextInputSequence, Tuple[InputSequence, InputSequence]]
Embedding attempt failed: TextEncodeInput must be Union[TextInputSequence, Tuple[InputSequence, InputSequence]]


TextEncodeInput must be Union[TextInputSequence, Tuple[InputSequence, InputSequence]] Unable to chunk


In [ ]:
from llama_index.llms.huggingface import HuggingFaceLLM
import os
from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
load_dotenv()

quants = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
)

model2 = HuggingFaceLLM(model_name="Qwen/Qwen2.5-3B-Instruct", tokenizer_name="Qwen/Qwen2.5-3B-Instruct",
                        model_kwargs={'quantization_config':quants}, device_map='auto',
                        context_window=3000,max_new_tokens=1)

In [ ]:
def custom_tag_classification(chunk):

    prompt = f"This RAG pipeline requires the document corpus to be chunked. Each chunk either has the tag of paper_body (0), or bibliography (1) to classify them. \n Here is the chunk: \n {chunk.text} \n You need to ONLY return 0 if its a paper_body, or return 1 if its a bibliography."
    
    response = model2.complete(prompt)

    chunk.metadata["classification"] = int(response.text);
    return chunk
    

    


In [30]:
import pickle
with open("nodes.pkl", "wb") as f:
    pickle.dump(nodes, f)